# RLM vs a plain LLM call: build an Excel report from 140 MB of IMF data

One task, two very different attempts:

- **gpt-5.1, plain call**: the flagship model gets the question plus as much raw
  CSV as fits in a prompt, and must answer directly.
- **gpt-5-mini through the RLM**: a model about 5x cheaper gets the same
  question plus an interpreter, and must build the actual deliverable: a
  formatted Excel workbook with a pivot of the 10 highest-inflation countries,
  a merged title cell, styled headers, and a second sheet with all countries.

The source is the IMF's CPI dataflow pulled live from their public SDMX API:
about 140 MB of CSV, 1.5 million observation rows, 194 countries. It does not
fit in any context window. A deterministic DuckDB query grades the result cell
by cell at the end.

In [ ]:
%pip install -q fabric-rlm[analytics] openpyxl

## Download the data

One GET against the IMF SDMX 3.0 API. No API key. About 140 MB; the IMF server
usually delivers it in under a minute. The cell skips the download if the file
is already there, and prefers Lakehouse `Files` when one is mounted.

In [ ]:
import os, urllib.request

DATA_DIR = "/lakehouse/default/Files" if os.path.isdir("/lakehouse/default/Files") else "."
DATA_PATH = os.path.join(DATA_DIR, "imf_cpi.csv")
URL = (
    "https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/CPI/5.0.0/*.*.*.*.*"
    "?c%5BTIME_PERIOD%5D=ge:2017-01-01+le:2026-12-31"
)

if not os.path.exists(DATA_PATH):
    req = urllib.request.Request(URL, headers={"Accept": "application/vnd.sdmx.data+csv"})
    with urllib.request.urlopen(req, timeout=900) as r, open(DATA_PATH, "wb") as fh:
        while chunk := r.read(1 << 20):
            fh.write(chunk)
print(f"{os.path.getsize(DATA_PATH) / 1e6:.0f} MB at {DATA_PATH}")

## Pick the models

In a Fabric notebook, `FabricLM` uses the capacity's built-in Azure OpenAI
endpoint: no key and nothing to provision. Check the hosted model list at
https://learn.microsoft.com/en-us/fabric/data-science/ai-services/ai-services-overview#consumption-rate

In [ ]:
from fabric_rlm import FabricLM

lm_big = FabricLM("gpt-5.1")       # the flagship, for the plain call
lm_mini = FabricLM("gpt-5-mini")   # the cheap model, for the RLM

# Outside Fabric, use any dspy.LM instead, for example:
# import dspy, os
# lm_big = dspy.LM("openrouter/openai/gpt-5.1", api_key=os.environ["OPENROUTER_API_KEY"],
#                  api_base="https://openrouter.ai/api/v1", max_tokens=16000, temperature=1.0)
# lm_mini = dspy.LM("openrouter/openai/gpt-5-mini", api_key=os.environ["OPENROUTER_API_KEY"],
#                   api_base="https://openrouter.ai/api/v1", max_tokens=16000, temperature=1.0)

## The task

In [ ]:
REPORT_PATH = os.path.join(DATA_DIR, "cpi_report.xlsx")

TASK = """You are given IMF CPI data in SDMX-CSV format (one observation per row)
at data_file. Build an Excel report at report_path (create the file with openpyxl).

Data selection: rows with INDEX_TYPE='CPI', COICOP_1999='_T',
TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT', FREQUENCY='M'. TIME_PERIOD looks like
'2021-M01'; COUNTRY is an ISO3 code; the value is in OBS_VALUE. A qualifying
country has all 60 monthly observations for 2021 through 2025.

The workbook has two sheets.

Sheet 'Report':
- A1:G1 merged, containing exactly: Average year-over-year CPI inflation (%), 2021-2025
- Row 2 headers, bold, with a solid light-gray fill (PatternFill fgColor D9D9D9):
  Country, 2021, 2022, 2023, 2024, 2025, Avg 2021-2025
- Set column A width to 32.
- Rows 3 to 12: the 10 qualifying countries with the highest five-year average of
  their monthly YoY values, sorted descending by that average. Each year column is
  that calendar year's average of the 12 monthly values; 'Avg 2021-2025' is the
  average of all 60 monthly values. Round to 2 decimals and apply number format
  '0.00' to every numeric cell.
- Row 13: column A = 'Median (all qualifying countries)', column G = the median
  across ALL qualifying countries of their five-year averages, 2 decimals,
  number format '0.00'.

Sheet 'All countries':
- Row 1 headers, bold: Country, Avg 2021-2025
- One row per qualifying country, sorted descending by the five-year average,
  values rounded to 2 decimals with number format '0.00'.

Save to report_path. Then reload the saved file with openpyxl and verify EVERY
requirement above: both sheet names exactly as given, the merge, bold headers
with the D9D9D9 fill, the column width, the '0.00' number formats, the sorted
rows, and the 'All countries' row count. Fix and re-save until every check
passes. Only then SUBMIT with n_countries (the count of qualifying countries)
and median_avg (the row-13 median value)."""

## Arm 1: gpt-5.1, plain call

The flagship model, 200,000 characters of the file (under 0.15 percent), and
the question. It cannot create a file, so it only has to report the numbers
that would go into the workbook. When we ran this, it burned about 109,000
prompt tokens and answered honestly that the task is impossible from the
slice: the visible rows do not even include the required series.

In [ ]:
import time

PLAIN_PROMPT = """The text below is the beginning of a large CSV of IMF CPI data in
SDMX-CSV format (one observation per row). The full file is about 140 MB; only
this slice fits in your context.

Rows with INDEX_TYPE='CPI', COICOP_1999='_T', TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT',
FREQUENCY='M' are monthly year-over-year all-items CPI inflation. TIME_PERIOD looks
like '2021-M01'; COUNTRY is an ISO3 code; the value is in OBS_VALUE. A qualifying
country has all 60 monthly observations for 2021 through 2025.

Report, as JSON only:
{"n_countries": <count of qualifying countries>,
 "top10": [[country, avg2021, avg2022, avg2023, avg2024, avg2025, avg_5yr], ...],
 "median_avg": <median across all qualifying countries of their 5-year averages>}
top10 = the 10 qualifying countries with the highest 5-year average of monthly YoY
values, sorted descending, yearly values = that year's average of 12 monthly values,
all numbers to 2 decimals."""

head = open(DATA_PATH, encoding="utf-8").read(200_000)
t0 = time.time()
plain_text = lm_big(f"{PLAIN_PROMPT}\n\n--- FILE SLICE ---\n{head}")[0]
plain_seconds = time.time() - t0
plain_usage = lm_big.history[-1]["usage"]
print(plain_text[:1500])

## Arm 2: gpt-5-mini through the RLM

The cheap model, the same question, and an interpreter. It writes DuckDB and
openpyxl code in the subprocess: aggregate 1.5 million rows, pivot by year,
merge and style the header, write both sheets, save the workbook.

In [ ]:
from fabric_rlm import File, RLM

t0 = time.time()
rlm = RLM.task(
    task=TASK,
    inputs={"data_file": File(DATA_PATH), "report_path": REPORT_PATH},
    outputs=["n_countries", "median_avg"],
    lm=lm_mini,
    skills=["data_exploration", "excel_modify"],
    max_turns=10,
    timeout=600.0,
)
result = rlm.run()
rlm_seconds = time.time() - t0
result.payload

## Ground truth

A deterministic DuckDB pivot computes the true report from the same file.

In [ ]:
import duckdb, statistics

con = duckdb.connect()
rows = con.execute(f"""
WITH obs AS (
    SELECT COUNTRY, substr(TIME_PERIOD, 1, 4) AS yr, OBS_VALUE
    FROM read_csv_auto('{DATA_PATH}')
    WHERE INDEX_TYPE = 'CPI' AND COICOP_1999 = '_T'
      AND TYPE_OF_TRANSFORMATION = 'YOY_PCH_PA_PT' AND FREQUENCY = 'M'
      AND substr(TIME_PERIOD, 1, 4) BETWEEN '2021' AND '2025'
      AND OBS_VALUE IS NOT NULL
), complete AS (
    SELECT COUNTRY FROM obs GROUP BY COUNTRY HAVING count(*) = 60
), yearly AS (
    SELECT o.COUNTRY, yr, avg(OBS_VALUE) AS y_avg
    FROM obs o JOIN complete c USING (COUNTRY) GROUP BY o.COUNTRY, yr
), fivey AS (
    SELECT o.COUNTRY, avg(OBS_VALUE) AS avg5
    FROM obs o JOIN complete c USING (COUNTRY) GROUP BY o.COUNTRY
)
SELECT f.COUNTRY,
  max(CASE WHEN yr = '2021' THEN y_avg END), max(CASE WHEN yr = '2022' THEN y_avg END),
  max(CASE WHEN yr = '2023' THEN y_avg END), max(CASE WHEN yr = '2024' THEN y_avg END),
  max(CASE WHEN yr = '2025' THEN y_avg END), max(avg5)
FROM fivey f JOIN yearly y ON f.COUNTRY = y.COUNTRY
GROUP BY f.COUNTRY ORDER BY max(avg5) DESC
""").fetchall()

truth = {
    "n_countries": len(rows),
    "top10": [[r[0]] + [round(v, 2) for v in r[1:]] for r in rows[:10]],
    "median_avg": round(statistics.median(r[6] for r in rows), 2),
}
truth["top10"]

## Grade the workbook

Reload the workbook with openpyxl and check it structurally (merged title,
bold and filled headers, column width, number formats, both sheets) and
numerically (all 60 pivot values, the median, and the full country list,
within 0.02).

In [ ]:
from openpyxl import load_workbook

# OpenAI list prices, USD per 1M tokens (input, output), for the cost column.
PRICE = {"gpt-5.1": (1.25, 10.00), "gpt-5-mini": (0.25, 2.00)}

def cost(model, prompt_tokens, completion_tokens):
    p_in, p_out = PRICE[model]
    return (prompt_tokens * p_in + completion_tokens * p_out) / 1e6

def ok(a, b):
    return abs(float(a) - b) <= 0.02

wb = load_workbook(REPORT_PATH)
ws = wb["Report"]
allc = wb["All countries"]
top = truth["top10"]

checks = {
    "A1_G1_merged": "A1:G1" in [str(r) for r in ws.merged_cells.ranges],
    "title": ws["A1"].value == "Average year-over-year CPI inflation (%), 2021-2025",
    "headers_bold": all(ws.cell(row=2, column=c).font.bold for c in range(1, 8)),
    "header_fill": str(ws["A2"].fill.fgColor.rgb).endswith("D9D9D9"),
    "col_A_width": ws.column_dimensions["A"].width >= 30,
    "number_format": ws["B3"].number_format == "0.00",
    "top10_countries": [ws.cell(row=3 + i, column=1).value for i in range(10)] == [r[0] for r in top],
    "top10_values": all(ok(ws.cell(row=3 + i, column=2 + j).value, top[i][1 + j]) for i in range(10) for j in range(6)),
    "median_row": ok(ws["G13"].value, truth["median_avg"]),
    "all_countries_rows": allc.max_row == truth["n_countries"] + 1,
    "all_countries_sorted": allc["A2"].value == top[0][0],
}
checks

In [ ]:
plain_correct = all(r[0] in plain_text for r in truth["top10"])
plain_tokens = plain_usage["prompt_tokens"] + plain_usage["completion_tokens"]
plain_cost = cost("gpt-5.1", plain_usage["prompt_tokens"], plain_usage["completion_tokens"])

rlm_correct = all(checks.values())
rlm_tokens = result.total_prompt_tokens + result.total_completion_tokens
rlm_cost = cost("gpt-5-mini", result.total_prompt_tokens, result.total_completion_tokens)

print(f"{'arm':<24} {'workbook':<10} {'tokens':>9} {'cost':>8} {'seconds':>9}")
print(f"{'plain call, gpt-5.1':<24} {'no file':<10} {plain_tokens:>9,} {plain_cost:>7.3f}$ {plain_seconds:>9.1f}")
print(f"{'RLM, gpt-5-mini':<24} {str(rlm_correct):<10} {rlm_tokens:>9,} {rlm_cost:>7.3f}$ {rlm_seconds:>9.1f}")

## What just happened

The flagship model failed: not for lack of intelligence, but because the data
was never in its context, and no amount of model quality fixes missing data.
The mini model succeeded because the RLM let it run code against all 1.5
million rows and write a real, formatted workbook, verified cell by cell
against ground truth. In our run the scoreboard read:

| arm | workbook | tokens | cost | seconds |
|---|---|---|---|---|
| plain call, gpt-5.1 | no file | 109,480 | $0.138 | 8.4 |
| RLM, gpt-5-mini | True | 47,642 | $0.023 | 78.4 |

The flagship spent six times more failing than the mini model spent
succeeding. That is the RLM trade: when the answer must be computed from data
that does not fit in context, an interpreter beats a bigger model. The README section "When to use
an RLM (and when not to)" covers when the trade is worth it.